# 03 · Model testing (June, out of time)
Evaluates both models on June, which no model has seen. Calibrates the probabilities on May, compares the agent prioritisation against today's rule (highest days-past-due first), explains the drivers and checks the acceptance criteria. Saves **artifact 6** (comparison) and **artifact 7** (charts).

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

import pipeline_config as cfg

df = pd.read_csv(cfg.CLEAN_FILE)
split = pd.read_csv(cfg.SPLIT_FILE).set_index("account_id")["split"]
df["split"] = df["account_id"].map(split)
features = cfg.feature_columns(df.drop(columns="split"))
calib_df, test_df = df[df["split"] == "calibrate"], df[df["split"] == "test"].copy()
models = {name: joblib.load(cfg.MODEL_DIR / f"{name}.joblib") for name in ["baseline_logreg", "gradient_boosting"]}
y_test = test_df[cfg.TARGET]
print(f"June test cohort: {len(test_df):,} accounts, pay rate {y_test.mean():.1%} | models: {list(models)}")

## 1. June metrics (raw probabilities)

In [ ]:
raw_scores = {name: m.predict_proba(test_df[features])[:, 1] for name, m in models.items()}
comparison = pd.DataFrame([{"model": name, "roc_auc": roc_auc_score(y_test, s),
                            "pr_auc": average_precision_score(y_test, s),
                            "brier": brier_score_loss(y_test, s)} for name, s in raw_scores.items()]).set_index("model")
comparison.round(3)

## 2. Calibration (fitted on May, checked on June)

In [ ]:
best = "gradient_boosting"
calibrator = IsotonicRegression(out_of_bounds="clip")
calibrator.fit(models[best].predict_proba(calib_df[features])[:, 1], calib_df[cfg.TARGET])
joblib.dump(calibrator, cfg.MODEL_DIR / "calibrator.joblib")

p_raw = raw_scores[best]
p_cal = calibrator.predict(p_raw)
calibration = pd.DataFrame({
    "mean_predicted": [p_raw.mean(), p_cal.mean()],
    "actual_pay_rate": [y_test.mean(), y_test.mean()],
    "brier": [brier_score_loss(y_test, p_raw), brier_score_loss(y_test, p_cal)],
}, index=["raw", "calibrated"])

fig, ax = plt.subplots(figsize=(5, 5))
for label, p in [("raw", p_raw), ("calibrated on May", p_cal)]:
    frac, mean_pred = calibration_curve(y_test, p, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac, marker="o", label=label)
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", linewidth=1)
ax.set(xlabel="Predicted probability of paying", ylabel="Actual pay rate", title="Calibration - June cohort")
ax.legend()
fig.tight_layout()
fig.savefig(cfg.CHART_DIR / "calibration.png", dpi=150)
plt.show()
calibration.round(4)

## 3. Agent prioritisation vs today's rule
Agents can work 37% of the cohort. Dispute and hardship accounts go to a specialist team. Success = share of the June cohort's **actual unpaid amount** that the agent list covers.

In [ ]:
test_df["p_pay"] = p_cal
test_df["expected_unpaid"] = test_df["p_pay"] * test_df["amount_due"]
test_df["actual_unpaid"] = (1 - test_df[cfg.TARGET]) * test_df["amount_due"]

capacity = round(cfg.AGENT_CAPACITY_SHARE * len(test_df))
eligible = test_df[(test_df[cfg.SPECIALIST_FLAGS] == 0).all(axis=1)]
total_unpaid = eligible["actual_unpaid"].sum()

def coverage(ranking_column):
    chosen = eligible.nlargest(capacity, ranking_column)
    return chosen["actual_unpaid"].sum() / total_unpaid

prioritisation = pd.Series({
    "today: highest days_past_due first": coverage("days_past_due"),
    "model: highest expected unpaid first": coverage("expected_unpaid"),
})
print(f"capacity {capacity:,} of {len(test_df):,} June accounts | eligible {len(eligible):,}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(prioritisation.index, prioritisation.values, color=["#9aa5b1", "#2f6db5"])
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set(xlabel="Share of June unpaid amount covered by the agent list", title="Agent prioritisation - June")
fig.tight_layout()
fig.savefig(cfg.CHART_DIR / "prioritisation.png", dpi=150)
plt.show()
prioritisation.map("{:.1%}".format)

## 4. Drivers (SHAP)

In [ ]:
pipe = models[best]
preprocess, booster = pipe[:-1], pipe[-1]
sample = test_df[features].sample(1000, random_state=cfg.RANDOM_STATE)
X_sample = pd.DataFrame(preprocess.transform(sample), columns=preprocess.get_feature_names_out())
shap_values = shap.TreeExplainer(booster).shap_values(X_sample)

shap.summary_plot(shap_values, X_sample, max_display=12, show=False)
plt.title("Top drivers of paying within 30 days")
plt.tight_layout()
plt.savefig(cfg.CHART_DIR / "shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Acceptance check and save

In [ ]:
checks = pd.DataFrame([
    {"criterion": f"June ROC-AUC >= {cfg.ACCEPTANCE['roc_auc']}", "value": comparison.loc[best, "roc_auc"],
     "pass": comparison.loc[best, "roc_auc"] >= cfg.ACCEPTANCE["roc_auc"]},
    {"criterion": "PR-AUC beats logistic baseline", "value": comparison.loc[best, "pr_auc"],
     "pass": comparison.loc[best, "pr_auc"] > comparison.loc["baseline_logreg", "pr_auc"]},
    {"criterion": "model list covers more unpaid than today's rule", "value": prioritisation.iloc[1],
     "pass": prioritisation.iloc[1] > prioritisation.iloc[0]},
    {"criterion": "calibration gap smaller after calibration",
     "value": abs(calibration.loc["calibrated", "mean_predicted"] - y_test.mean()),
     "pass": abs(calibration.loc["calibrated", "mean_predicted"] - y_test.mean())
             < abs(calibration.loc["raw", "mean_predicted"] - y_test.mean())},
])

summary = comparison.copy()
summary.loc[best, "brier_calibrated"] = calibration.loc["calibrated", "brier"]
summary.loc[best, "coverage_model"] = prioritisation.iloc[1]
summary.loc[best, "coverage_dpd_rule"] = prioritisation.iloc[0]
summary.to_csv(cfg.COMPARISON_FILE)
checks.round(3)